# Paper #31 Implementation: Global Solar Magnetic Fields / 전역 태양 자기장

Based on Mackay & Yeates (2012), *Living Reviews in Solar Physics*, DOI: 10.12942/lrsp-2012-6.

이 노트북은 **전역 광구/코로나 자기장 모델링의 핵심 개념**을 구현한다:
This notebook implements the **core concepts of global photospheric/coronal magnetic field modeling**:

1. Potential field source surface (PFSS) 모델 / PFSS model
2. 구면 조화 분해 / Spherical harmonic decomposition
3. 합성 synoptic magnetogram / Synthetic synoptic magnetogram
4. 1D 표면 플럭스 수송 (advection + diffusion) / 1D surface flux transport
5. 태양 주기에 따른 쌍극자 모멘트 진화 / Dipole moment evolution across the solar cycle

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import sph_harm, lpmv

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

R_SUN = 1.0
R_SS = 2.5

## 1. PFSS Radial Falloff for Each Harmonic Degree / 조화 차수별 PFSS 반경 감쇠

포텐셜 장을 구면 조화로 전개하면 각 $(\ell, m)$ 모드의 반경 의존성은 다음과 같다:

For a potential field expanded in spherical harmonics, the radial dependence of each $(\ell, m)$ mode is:

$$
B_r^{\ell m}(r) \propto c_\ell(r) = \frac{\ell r^{\ell-1} + (\ell+1) R_{ss}^{-2\ell-1} r^{-\ell-2}}{\ell + (\ell+1)(R_\odot/R_{ss})^{2\ell+1}} \cdot \text{(photospheric coeff)}
$$

경계 조건: $B_r$는 $r=R_\odot$에서 주어지고, $B_\theta = B_\phi = 0$는 $r=R_{ss}$에서 성립.
Boundary conditions: $B_r$ given at $r=R_\odot$, $B_\theta=B_\phi=0$ at $r=R_{ss}$.

높은 $\ell$일수록 source surface에서 훨씬 더 빠르게 감쇠한다 — 이는 소규모 구조가 근거리에서만 지배적이라는 관측과 일치.
Higher $\ell$ decays much faster toward the source surface — consistent with the observation that small-scale features dominate only at low altitudes.

In [ ]:
def pfss_radial_factor(ell, r, R_ss=R_SS, R_star=R_SUN):
    """Radial PFSS factor for harmonic degree ell.

    Returns the ratio B_r(r) / B_r(R_star) for a single (ell, m) mode
    under the PFSS boundary conditions.

    Args:
        ell: Harmonic degree (integer >= 1).
        r: Radius array (in units of R_star).
        R_ss: Source surface radius.
        R_star: Stellar surface radius.

    Returns:
        Array of the same shape as r with the radial factor.
    """
    num = ell * (r / R_star) ** (ell - 1) + (ell + 1) * (R_ss ** (-(2 * ell + 1))) * r ** (-ell - 2)
    denom = ell + (ell + 1) * (R_star / R_ss) ** (2 * ell + 1)
    return num / denom

r = np.linspace(1.0, R_SS, 200)
plt.figure()
for ell in [1, 2, 3, 5, 10]:
    plt.plot(r, pfss_radial_factor(ell, r), label=f'$\\ell$ = {ell}')
plt.axvline(R_SS, color='k', linestyle='--', alpha=0.5, label=f'$R_{{ss}} = {R_SS} R_\\odot$')
plt.yscale('log')
plt.xlabel('r / R$_\\odot$')
plt.ylabel('B$_r$(r) / B$_r$(R$_\\odot$)  (log scale)')
plt.title('PFSS radial falloff by harmonic degree / 조화 차수별 PFSS 반경 감쇠')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

for ell in [1, 2, 5, 10]:
    frac = pfss_radial_factor(ell, R_SS) / pfss_radial_factor(ell, 1.0)
    print(f'ell = {ell:2d}: B_r(R_ss) / B_r(R_sun) = {frac:.4e}')

## 2. Simple Dipole + Quadrupole Potential Field / 단순 쌍극자 + 사극자 포텐셜 장

축대칭 (m=0) 쌍극자와 사극자 성분을 조합하여 광구 경계 조건을 만든다. 쌍극자는 1/r³, 사극자는 1/r⁴로 감쇠.
Combine axisymmetric (m=0) dipole and quadrupole components as photospheric BC. Dipole falls as 1/r³, quadrupole as 1/r⁴.

포텐셜: $\Psi = \sum_\ell A_\ell r^{-\ell-1} P_\ell(\cos\theta)$ (PFSS 안에서는 내부/외부 항 모두 포함하지만 여기서는 단순화).
Potential: $\Psi = \sum_\ell A_\ell r^{-\ell-1} P_\ell(\cos\theta)$ (simplified — full PFSS has both interior and exterior terms).

In [ ]:
def axisymmetric_pfss_Br(r, theta, coeffs, R_ss=R_SS, R_star=R_SUN):
    """Axisymmetric (m=0) PFSS B_r.

    Args:
        r: Radius (scalar or array, units of R_star).
        theta: Colatitude (radians, same shape as r or broadcastable).
        coeffs: Dict {ell: amplitude} of photospheric B_r coefficients in the P_ell basis.

    Returns:
        B_r at (r, theta) in units of the input amplitude.
    """
    result = np.zeros_like(r + theta)
    for ell, A in coeffs.items():
        P_ell = lpmv(0, ell, np.cos(theta))
        result = result + A * pfss_radial_factor(ell, r, R_ss, R_star) * P_ell
    return result

n_r, n_t = 120, 120
r_grid = np.linspace(1.01, R_SS, n_r)
theta_grid = np.linspace(0.01, np.pi - 0.01, n_t)
R, THETA = np.meshgrid(r_grid, theta_grid, indexing='ij')

coeffs = {1: 5.0, 2: 1.5}
Br = axisymmetric_pfss_Br(R, THETA, coeffs)

X = R * np.sin(THETA)
Y = R * np.cos(THETA)

fig, ax = plt.subplots(figsize=(8, 8))
cf = ax.pcolormesh(X, Y, Br, cmap='RdBu_r',
                   vmin=-np.nanmax(np.abs(Br)), vmax=np.nanmax(np.abs(Br)),
                   shading='auto')
theta_circ = np.linspace(0, np.pi, 180)
ax.plot(np.sin(theta_circ), np.cos(theta_circ), 'k-', lw=2, label='Photosphere')
ax.plot(R_SS * np.sin(theta_circ), R_SS * np.cos(theta_circ), 'k--', lw=1.5,
        label=f'Source surface $R_{{ss}}={R_SS}$')
ax.set_aspect('equal')
ax.set_xlabel('x / R$_\\odot$')
ax.set_ylabel('z / R$_\\odot$')
ax.set_title('Dipole + Quadrupole PFSS B$_r$ / 쌍극자+사극자 PFSS')
ax.legend()
plt.colorbar(cf, ax=ax, label='B$_r$ (arb. units)')
plt.show()

## 3. Synthetic Synoptic Magnetogram / 합성 시놉틱 자기지도

카링턴 경도 × 위도 격자에 몇 개의 쌍극 활동영역을 배치한 합성 광구 자기지도를 만든다. 이것이 전역 모델의 입력 경계조건이 된다.
Build a synthetic photospheric B_r map on a Carrington longitude × latitude grid with a handful of bipolar active regions. This is the input BC for global models.

In [ ]:
def bipolar_region(lon_grid, lat_grid, lon0, lat0, tilt_deg, sep_deg, strength):
    """Add a Gaussian bipolar active region to the synoptic grid.

    Args:
        lon_grid, lat_grid: 2D grids (degrees).
        lon0, lat0: Center of the region (degrees).
        tilt_deg: Joy's-law tilt angle of the bipole.
        sep_deg: Polarity separation.
        strength: Peak field strength (Gauss).

    Returns:
        2D array of B_r contribution.
    """
    tilt = np.deg2rad(tilt_deg)
    dlon = 0.5 * sep_deg * np.cos(tilt)
    dlat = 0.5 * sep_deg * np.sin(tilt)
    sigma = 0.3 * sep_deg
    pos = np.exp(-((lon_grid - lon0 - dlon) ** 2 + (lat_grid - lat0 - dlat) ** 2) / (2 * sigma ** 2))
    neg = np.exp(-((lon_grid - lon0 + dlon) ** 2 + (lat_grid - lat0 + dlat) ** 2) / (2 * sigma ** 2))
    return strength * (pos - neg)

lon = np.linspace(0, 360, 361)
lat = np.linspace(-90, 90, 181)
LON, LAT = np.meshgrid(lon, lat)

Br_map = np.zeros_like(LON)
ar_list = [
    (60, 15, 5, 10, 200),
    (180, -12, -5, 12, -250),
    (270, 20, 8, 8, 180),
    (340, -22, -9, 9, -150),
]
for (lon0, lat0, tilt, sep, strength) in ar_list:
    Br_map += bipolar_region(LON, LAT, lon0, lat0, tilt, sep, strength)

Br_map += 5 * np.sin(np.deg2rad(LAT)) * np.exp(-(np.abs(LAT) - 70) ** 2 / 200)

plt.figure(figsize=(12, 5))
vmax = np.max(np.abs(Br_map))
plt.pcolormesh(lon, lat, Br_map, cmap='RdBu_r', vmin=-vmax, vmax=vmax, shading='auto')
plt.colorbar(label='B$_r$ (G)')
plt.xlabel('Carrington longitude (deg)')
plt.ylabel('Latitude (deg)')
plt.title('Synthetic synoptic magnetogram / 합성 시놉틱 자기지도')
plt.show()

print(f'Total signed flux (arb): {np.sum(Br_map) * np.cos(np.deg2rad(LAT)).mean():.2e}')
print(f'Unsigned flux (arb):      {np.sum(np.abs(Br_map)) * np.cos(np.deg2rad(LAT)).mean():.2e}')

## 4. Spherical Harmonic Decomposition / 구면 조화 분해

합성 자기지도를 구면 조화 계수 $a_{\ell m}$로 분해:
Decompose the synoptic map into spherical harmonic coefficients $a_{\ell m}$:

$$
B_r(\theta, \phi) = \sum_{\ell=0}^{\ell_{\max}} \sum_{m=-\ell}^{\ell} a_{\ell m} Y_{\ell m}(\theta, \phi)
$$

실수값 $B_r$에 대해 직접 적분으로 $a_{\ell m}$를 계산한다. 저차 조화의 파워 스펙트럼은 쌍극자/사극자 성분이 어떻게 합쳐져 있는지 보여준다.
For real $B_r$, compute $a_{\ell m}$ by direct integration. The low-degree power spectrum shows how dipole/quadrupole components are combined.

In [ ]:
def compute_alm(Br, lat_deg, lon_deg, ell_max=8):
    """Compute spherical harmonic coefficients of a real B_r map.

    Uses numerical integration over the (theta, phi) grid with the
    sin(theta) Jacobian.

    Args:
        Br: 2D array (n_lat, n_lon).
        lat_deg, lon_deg: 1D grids in degrees.
        ell_max: Max harmonic degree.

    Returns:
        Dict {(ell, m): complex coefficient} and a power spectrum array.
    """
    theta = np.deg2rad(90.0 - lat_deg)
    phi = np.deg2rad(lon_deg)
    PHI, THETA = np.meshgrid(phi, theta)
    dtheta = theta[1] - theta[0]
    dphi = phi[1] - phi[0]
    sin_t = np.sin(THETA)
    alm = {}
    power = np.zeros(ell_max + 1)
    for ell in range(ell_max + 1):
        for m in range(-ell, ell + 1):
            Y = sph_harm(m, ell, PHI, THETA)
            coef = np.sum(Br * np.conj(Y) * sin_t) * dtheta * dphi
            alm[(ell, m)] = coef
            power[ell] += np.abs(coef) ** 2
    return alm, power

alm, power = compute_alm(Br_map, lat, lon, ell_max=8)

plt.figure()
plt.bar(np.arange(len(power)), power)
plt.yscale('log')
plt.xlabel('Harmonic degree $\\ell$')
plt.ylabel('Power $\\sum_m |a_{\\ell m}|^2$')
plt.title('Angular power spectrum of synoptic map / 시놉틱 자기지도 각 파워 스펙트럼')
plt.grid(alpha=0.3)
plt.show()

dipole_amp = np.sqrt(sum(np.abs(alm[(1, m)]) ** 2 for m in range(-1, 2)))
quad_amp = np.sqrt(sum(np.abs(alm[(2, m)]) ** 2 for m in range(-2, 3)))
print(f'Dipole amplitude (ell=1):     {dipole_amp:.3e}')
print(f'Quadrupole amplitude (ell=2): {quad_amp:.3e}')
print(f'Dipole : Quadrupole ratio    ~ {dipole_amp / quad_amp:.2f}')

## 5. 1D Surface Flux Transport / 1D 표면 플럭스 수송

광구 $B_r$의 대규모 진화는 위도 방향의 자오면 유동(advection), 초과립 난류(diffusion), 활동영역 출현(source term)으로 근사된다:

The large-scale evolution of photospheric $B_r$ is well approximated by meridional flow (advection), supergranular turbulence (diffusion), and active region emergence (source):

$$
\frac{\partial B_r}{\partial t} = -\frac{1}{R_\odot \cos\lambda} \frac{\partial}{\partial \lambda}\left[v(\lambda) \cos\lambda\, B_r\right] + \frac{\eta}{R_\odot^2 \cos\lambda} \frac{\partial}{\partial\lambda}\left[\cos\lambda \frac{\partial B_r}{\partial\lambda}\right] + S(\lambda, t)
$$

여기서 $\lambda$는 위도, $v(\lambda)$는 자오면 유동(극 방향으로 10-20 m/s), $\eta \sim 600$ km²/s는 난류 확산.
Here $\lambda$ is latitude, $v(\lambda)$ is the poleward meridional flow (10-20 m/s), and $\eta \sim 600$ km²/s is the turbulent diffusion.

In [ ]:
def meridional_flow(lat_rad, v0=15.0):
    """Poleward meridional flow profile.

    Args:
        lat_rad: Latitude in radians.
        v0: Peak flow speed (m/s).

    Returns:
        Flow velocity in m/s (positive = poleward from equator).
    """
    return v0 * np.sin(2 * lat_rad) * np.cos(lat_rad) ** 0

def step_sft(Br, lat, dt, eta, v0):
    """One explicit Euler step of the 1D surface flux transport equation.

    Args:
        Br: 1D array of B_r values at latitudes `lat`.
        lat: Latitude array in radians (uniform grid).
        dt: Timestep (seconds).
        eta: Turbulent diffusivity (m^2/s).
        v0: Peak meridional flow (m/s).

    Returns:
        Updated B_r.
    """
    R = 6.96e8
    dlat = lat[1] - lat[0]
    cos_l = np.cos(lat)
    v = meridional_flow(lat, v0)
    flux = v * cos_l * Br
    dflux = np.gradient(flux, dlat)
    adv = -1.0 / (R * cos_l) * dflux
    dBr_dlat = np.gradient(Br, dlat)
    diff_flux = cos_l * dBr_dlat
    d_diff = np.gradient(diff_flux, dlat)
    diff = eta / (R ** 2 * cos_l) * d_diff
    return Br + dt * (adv + diff)

lat_rad = np.linspace(-np.pi / 2 + 0.02, np.pi / 2 - 0.02, 181)
Br_lat = np.zeros_like(lat_rad)
for sign, center in [(+1, +0.3), (-1, +0.25), (-1, -0.3), (+1, -0.25)]:
    Br_lat += sign * 100 * np.exp(-((lat_rad - center) ** 2) / (2 * 0.03 ** 2))
Br_lat_init = Br_lat.copy()

dt = 3600.0
eta = 600e6
v0 = 15.0

snapshots = {0: Br_lat.copy()}
for days in [30, 180, 365, 730]:
    n_steps = int(days * 86400 / dt)
    Br_now = Br_lat.copy() if days == 30 else snapshots[prev_days].copy()
    start_step = 0 if days == 30 else int(prev_days * 86400 / dt)
    for _ in range(n_steps - start_step):
        Br_now = step_sft(Br_now, lat_rad, dt, eta, v0)
    snapshots[days] = Br_now
    prev_days = days

plt.figure(figsize=(11, 6))
for days, profile in snapshots.items():
    plt.plot(np.rad2deg(lat_rad), profile, label=f't = {days} d')
plt.axhline(0, color='k', lw=0.5)
plt.xlabel('Latitude (deg)')
plt.ylabel('B$_r$ (G)')
plt.title('Surface flux transport — meridional advection + supergranular diffusion / 표면 플럭스 수송')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 6. Dipole Moment Evolution Across an 11-Year Cycle / 11년 주기 동안의 쌍극자 모멘트 진화

Joy의 법칙 경향 (기울어진 쌍극성 활동영역)으로 인한 새 플럭스를 주입하면서 축 대칭 쌍극자 모멘트 $D = \int B_r \cos\lambda \, d\cos\lambda$의 진화를 추적한다. Babcock-Leighton 패러다임의 핵심: 활동영역 기울어짐 → 극성 영역에서 자기 플럭스의 상쇄/증강 → 다음 주기의 역전된 쌍극자.

Track the axial dipole moment $D = \int B_r \cos\lambda \, d\cos\lambda$ as new flux is injected with Joy's law tilt (tilted bipolar ARs). This is the heart of the Babcock-Leighton paradigm: AR tilt → cancellation/enhancement of polar flux → reversed dipole in the next cycle.

In [ ]:
def axial_dipole(Br, lat_rad):
    """Axial (ell=1, m=0) dipole moment of a latitudinal B_r profile.

    Args:
        Br: 1D array at latitudes `lat_rad`.
        lat_rad: Latitude array (radians).

    Returns:
        Scalar dipole moment (same units as Br).
    """
    cos_l = np.cos(lat_rad)
    sin_l = np.sin(lat_rad)
    return np.trapz(Br * sin_l * cos_l, lat_rad) / np.trapz(cos_l ** 2, lat_rad)

lat_rad = np.linspace(-np.pi / 2 + 0.02, np.pi / 2 - 0.02, 181)
Br = np.zeros_like(lat_rad)
Br += 3 * np.sign(lat_rad) * (np.abs(lat_rad) > np.deg2rad(60))

cycle_years = 11.0
dt = 86400.0
t_end = 11 * 365.25 * 86400.0
n_steps = int(t_end / dt)

rng = np.random.default_rng(seed=0)
dipole_history = []
t_history = []
next_ar_time = 30 * 86400.0

for step in range(n_steps):
    t = step * dt
    Br = step_sft(Br, lat_rad, dt, eta=600e6, v0=15.0)
    if t >= next_ar_time:
        cycle_phase = 2 * np.pi * (t / 86400.0 / 365.25) / cycle_years
        activity = np.sin(cycle_phase / 2) ** 2
        n_ar = rng.poisson(3 * activity) + 1
        for _ in range(n_ar):
            hemisphere = rng.choice([-1, 1])
            lat_center = hemisphere * np.deg2rad(30 - 20 * (cycle_phase / (2 * np.pi)))
            tilt = hemisphere * np.deg2rad(5 + rng.normal(0, 3))
            sep = np.deg2rad(8)
            strength = rng.uniform(50, 150)
            leading_lat = lat_center - 0.5 * sep * np.cos(tilt)
            trailing_lat = lat_center + 0.5 * sep * np.cos(tilt)
            sigma = np.deg2rad(3)
            Br += -hemisphere * strength * np.exp(-((lat_rad - leading_lat) ** 2) / (2 * sigma ** 2))
            Br += +hemisphere * strength * np.exp(-((lat_rad - trailing_lat) ** 2) / (2 * sigma ** 2))
        next_ar_time = t + rng.exponential(30 * 86400.0 / max(activity, 0.05))
    if step % 30 == 0:
        dipole_history.append(axial_dipole(Br, lat_rad))
        t_history.append(t / (365.25 * 86400))

t_history = np.array(t_history)
dipole_history = np.array(dipole_history)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
ax1.plot(t_history, dipole_history, 'b-', lw=1.2)
ax1.axhline(0, color='k', lw=0.5)
ax1.set_ylabel('Axial dipole moment (G)')
ax1.set_title('Dipole evolution under BL source + surface transport / BL 소스 + 표면 수송에서 쌍극자 진화')
ax1.grid(alpha=0.3)

t_cycle = np.linspace(0, 11, 1000)
activity_curve = np.sin(np.pi * t_cycle / 11) ** 2
ax2.plot(t_cycle, activity_curve, 'r-', lw=1.5)
ax2.set_xlabel('Time (years)')
ax2.set_ylabel('AR emergence rate (arb.)')
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Initial dipole:    {dipole_history[0]:+.2f} G')
print(f'Mid-cycle dipole:  {dipole_history[len(dipole_history) // 2]:+.2f} G')
print(f'Final dipole:      {dipole_history[-1]:+.2f} G')
print('\nBabcock-Leighton 핵심: Joy\'s-law 기울어진 AR이 선행 극성을 적도쪽으로, 후행 극성을 극쪽으로 수송 → 극성 역전.')
print('Babcock-Leighton key: tilted ARs transport leading polarity equatorward and trailing polarity poleward → polarity reversal.')

## Summary / 요약

이 노트북은 Mackay & Yeates (2012) 리뷰의 핵심 수치 도구를 재현했다:

This notebook reproduced the core numerical tools of the Mackay & Yeates (2012) review:

1. **PFSS 반경 감쇠** — 높은 $\ell$은 source surface에서 급격히 소멸 (~10^-5 배), 전역 open flux는 주로 쌍극자/사극자가 지배.
   **PFSS radial falloff** — high $\ell$ decays sharply by the source surface (factor ~10⁻⁵), so global open flux is dominated by dipole/quadrupole components.

2. **축대칭 PFSS** — 쌍극자+사극자 조합의 field-line 기하는 실제 코로나 홀/헬멧 스트리머 구조와 정성적으로 일치.
   **Axisymmetric PFSS** — dipole+quadrupole field-line geometry qualitatively matches real coronal hole / helmet streamer structure.

3. **합성 시놉틱** — 몇 개의 쌍극 AR과 극성 띠로 구성된 지도는 실제 관측 자기지도의 주요 특징을 재현.
   **Synthetic synoptic** — a few bipolar ARs plus polar bands reproduce the main features of real magnetograms.

4. **구면 조화 분해** — 저차 조화가 대부분의 signed flux를 담당; 활동영역은 $\ell \ge 3$에서만 나타남.
   **Spherical harmonic decomposition** — low-$\ell$ modes carry most of the signed flux; active regions live only at $\ell \ge 3$.

5. **1D SFT** — advection + diffusion으로 AR이 빠르게 확산되어 극 영역으로 수송됨을 보임.
   **1D SFT** — advection + diffusion show how ARs quickly spread and are transported to the polar regions.

6. **쌍극자 모멘트 진화** — Joy의 법칙 기울어짐 있는 합성 AR 주입이 Babcock-Leighton 기반 극성 역전 시뮬레이션.
   **Dipole evolution** — synthetic AR injection with Joy's-law tilt simulates the Babcock-Leighton-driven polarity reversal.

### References

- Mackay, D. & Yeates, A. (2012). *The Sun's Global Photospheric and Coronal Magnetic Fields: Observations and Models*. Living Reviews in Solar Physics, 9(6). https://doi.org/10.12942/lrsp-2012-6
- Schatten, Wilcox & Ness (1969). *A model of interplanetary and coronal magnetic fields*. Solar Physics, 6, 442.
- Altschuler, M. & Newkirk, G. (1969). *Magnetic fields and the structure of the solar corona*. Solar Physics, 9, 131.
- Wang, Y.-M. & Sheeley, N. R. Jr. (1991). *Magnetic flux transport and the sun's dipole moment*. ApJ, 375, 761.